# 02 — ZoomMoE: model and staged training

Implements the three components of the proposal on top of the folds built in
notebook 01:

* **A — Progressive Zoom Refinement (PZR).** An ordered residual-gated state
  `h_k = h_{k-1} + g_k * delta_k`, where queries derived from `h_{k-1}` attend
  into the spatial tokens of magnification `m_k`. Replaces the permutation-
  invariant set attention of the baseline.
* **B — Zoom-conditioned Conv-MoE inside the CNN.** Sparse top-k expert blocks
  in the last two ResNet stages. The router sees the local feature, the
  magnification embedding, **and `h_{k-1}`** — so expert choice at 400x depends
  on what the coarser views already found. That feedback edge is the main
  architectural claim.
* **C — Contrastive alignment.** Cross-magnification InfoNCE with patient-safe
  negatives, an invariant/specific split with a gradient-reversal layer, a
  zoom-ordinal metric regulariser, and SupCon with cross-patient positives.

Training runs in three stages: unsupervised warm-up, supervised fusion,
end-to-end fine-tune.

**Cost warning.** The model runs the backbone once per magnification, so one
sample is four ResNet-50 forward passes. Start with `QUICK = True` to verify
the pipeline on a patient subset before committing to a full 5-fold run.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# Add the folder containing breakhis_common.py
CODE_DIR = Path("/kaggle/input/datasets/nabinkoirala452/code-files")

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Import project utilities
import breakhis_common as bc

# Device
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Reproducibility
SEED = 42

# Output directory
OUT = Path("/kaggle/working/artifacts")
OUT.mkdir(parents=True, exist_ok=True)

# Set seed
bc.set_seed(SEED)

print("code directory:", CODE_DIR)
print("device:", DEVICE)
print("torch:", torch.__version__)

code directory: /kaggle/input/datasets/nabinkoirala452/code-files
device: cuda
torch: 2.10.0+cu128


## 1. Model definition

Written to `zoommoe_model.py` so notebook 03 imports exactly the same code.

In [2]:
%%writefile zoommoe_model.py
"""ZoomMoE: zoom-conditioned MoE backbone + progressive zoom refinement."""

from collections import defaultdict
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

N_MAGS = 4


@dataclass
class ZoomMoEConfig:
    img_size: int = 224
    n_mags: int = N_MAGS
    mag_dim: int = 32
    state_dim: int = 256
    tok_dim: int = 256
    cond_state_dim: int = 64
    n_queries: int = 8
    attn_heads: int = 4
    proj_dim: int = 128
    n_experts: int = 8
    top_k: int = 2
    expert_ratio: int = 4
    moe_stages: Tuple[str, ...] = ("layer3", "layer4")
    pretrained: bool = True
    dropout: float = 0.3
    # loss weights
    w_align: float = 1.0
    w_ordinal: float = 0.5
    w_disent: float = 0.5
    w_advmag: float = 0.2
    w_supcon: float = 0.5
    w_deep: float = 0.3
    w_bal: float = 0.01
    w_spec: float = 0.05
    w_zloss: float = 1e-3
    tau: float = 0.1
    ordinal_margin: float = 0.05


# ---------------------------------------------------------------- gradient reversal
class _GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad):
        return -ctx.lambd * grad, None


def grad_reverse(x, lambd=1.0):
    return _GradReverse.apply(x, lambd)


# ---------------------------------------------------------------- Conv-MoE
class ConvMoE(nn.Module):
    """
    Sparse top-k mixture of convolutional experts with per-location routing.

    The router input is [local feature ; conditioning], where conditioning is
    [magnification embedding ; projected zoom state h_{k-1}]. Experts are
    inverted bottlenecks so that running all of them densely stays affordable;
    a production version would gather tokens per expert instead.
    """

    def __init__(self, channels: int, cond_dim: int, n_experts: int = 8, top_k: int = 2, ratio: int = 4):
        super().__init__()
        self.n_experts, self.top_k = n_experts, top_k
        hidden = max(channels // ratio, 32)
        self.norm = nn.GroupNorm(1, channels)
        self.router = nn.Linear(channels + cond_dim, n_experts)
        self.experts = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Conv2d(channels, hidden, 1, bias=False),
                    nn.Conv2d(hidden, hidden, 3, padding=1, groups=hidden, bias=False),
                    nn.GELU(),
                    nn.Conv2d(hidden, channels, 1, bias=False),
                )
                for _ in range(n_experts)
            ]
        )
        nn.init.zeros_(self.router.bias)

    def forward(self, x: torch.Tensor, cond: torch.Tensor):
        B, C, H, W = x.shape
        N = H * W
        u = self.norm(x).permute(0, 2, 3, 1).reshape(B, N, C)
        c = cond.unsqueeze(1).expand(-1, N, -1)
        logits = self.router(torch.cat([u, c], dim=-1))

        if self.training:
            logits = logits + torch.randn_like(logits) / self.n_experts

        topv, topi = logits.topk(self.top_k, dim=-1)
        # FIX: Cast softmax back to the original dtype to avoid AMP scatter mismatch
        gates = torch.zeros_like(logits).scatter(-1, topi, topv.softmax(dim=-1).to(logits.dtype))

        out = x
        for e, expert in enumerate(self.experts):
            g = gates[..., e]
            if not self.training and float(g.detach().abs().sum()) == 0.0:
                continue
            out = out + g.reshape(B, 1, H, W) * expert(x)

        stats = {
            "gates": gates.mean(dim=(0, 1)),
            "frac": (gates > 0).float().mean(dim=(0, 1)),
            "zloss": logits.logsumexp(dim=-1).pow(2).mean(),
        }
        return out, stats

# ---------------------------------------------------------------- backbone
class MoEBackbone(nn.Module):
    """ResNet-50 with Conv-MoE blocks inserted at the end of the last stages."""

    def __init__(self, cond_dim: int, cfg: ZoomMoEConfig):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if cfg.pretrained else None
        r = models.resnet50(weights=weights)
        self.stem = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool)
        self.layer1, self.layer2, self.layer3, self.layer4 = r.layer1, r.layer2, r.layer3, r.layer4

        mk = lambda ch: ConvMoE(ch, cond_dim, cfg.n_experts, cfg.top_k, cfg.expert_ratio)
        self.moe3 = mk(1024) if "layer3" in cfg.moe_stages else None
        self.moe4 = mk(2048) if "layer4" in cfg.moe_stages else None
        self.out_channels = 2048

    def freeze_until(self, stage: Optional[str]):
        order = ["stem", "layer1", "layer2", "layer3", "layer4"]
        if stage is None:
            for p in self.parameters():
                p.requires_grad = True
            return
        stop = order.index(stage)
        for i, name in enumerate(order):
            for p in getattr(self, name).parameters():
                p.requires_grad = i > stop

    def forward(self, x: torch.Tensor, cond: torch.Tensor):
        stats = []
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        if self.moe3 is not None:
            x, s = self.moe3(x, cond)
            stats.append(s)
        x = self.layer4(x)
        if self.moe4 is not None:
            x, s = self.moe4(x, cond)
            stats.append(s)
        return x, stats


# ---------------------------------------------------------------- progressive zoom
class ProgressiveZoom(nn.Module):
    """h_k = h_{k-1} + g_k * delta_k, driven by cross-attention into F_{m_k}."""

    def __init__(self, cfg: ZoomMoEConfig):
        super().__init__()
        self.n_q, self.tok_dim = cfg.n_queries, cfg.tok_dim
        self.to_q = nn.Sequential(
            nn.Linear(cfg.state_dim, cfg.tok_dim),
            nn.GELU(),
            nn.Linear(cfg.tok_dim, cfg.n_queries * cfg.tok_dim),
        )
        self.attn = nn.MultiheadAttention(cfg.tok_dim, cfg.attn_heads, batch_first=True)
        self.ln_q = nn.LayerNorm(cfg.tok_dim)
        self.ln_t = nn.LayerNorm(cfg.tok_dim)
        joint = cfg.state_dim + cfg.tok_dim
        self.gate = nn.Sequential(nn.Linear(joint, cfg.state_dim), nn.Sigmoid())
        self.delta = nn.Sequential(
            nn.Linear(joint, cfg.state_dim), nn.GELU(), nn.Dropout(cfg.dropout),
            nn.Linear(cfg.state_dim, cfg.state_dim),
        )
        self.step_head = nn.Linear(cfg.state_dim, 1)

    def step(self, h: torch.Tensor, tokens: torch.Tensor):
        B = h.shape[0]
        q = self.ln_q(self.to_q(h).view(B, self.n_q, self.tok_dim))
        t = self.ln_t(tokens)
        a, attn_w = self.attn(q, t, t, need_weights=True)
        a = a.mean(dim=1)
        hc = torch.cat([h, a], dim=-1)
        upd = self.gate(hc) * self.delta(hc)
        h_new = h + upd
        utility = upd.abs().sum(dim=-1)
        return h_new, self.step_head(h_new).squeeze(-1), utility, attn_w


# ---------------------------------------------------------------- full model
class ZoomMoE(nn.Module):
    def __init__(self, cfg: ZoomMoEConfig):
        super().__init__()
        self.cfg = cfg
        cond_dim = cfg.mag_dim + cfg.cond_state_dim

        self.mag_emb = nn.Embedding(cfg.n_mags, cfg.mag_dim)
        # cond_state_dim == 0 removes the zoom-state feedback edge (ablation)
        self.state_to_cond = (
            nn.Sequential(
                nn.LayerNorm(cfg.state_dim), nn.Linear(cfg.state_dim, cfg.cond_state_dim), nn.GELU()
            )
            if cfg.cond_state_dim > 0
            else None
        )
        self.backbone = MoEBackbone(cond_dim, cfg)
        C = self.backbone.out_channels

        self.tok_proj = nn.Conv2d(C, cfg.tok_dim, 1)
        n_tok = (cfg.img_size // 32) ** 2
        self.pos = nn.Parameter(torch.zeros(1, n_tok, cfg.tok_dim))
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.zoom_tok = nn.Embedding(cfg.n_mags, cfg.tok_dim)

        self.pzr = ProgressiveZoom(cfg)
        self.h0 = nn.Parameter(torch.zeros(1, cfg.state_dim))
        self.head = nn.Sequential(nn.LayerNorm(cfg.state_dim), nn.Dropout(cfg.dropout), nn.Linear(cfg.state_dim, 1))

        mkproj = lambda: nn.Sequential(nn.Linear(C, 512), nn.GELU(), nn.Linear(512, cfg.proj_dim))
        self.proj_inv = mkproj()
        self.proj_spec = mkproj()
        self.mag_clf_inv = nn.Linear(cfg.proj_dim, cfg.n_mags)
        self.mag_clf_spec = nn.Linear(cfg.proj_dim, cfg.n_mags)

    def forward(
        self,
        images: torch.Tensor,          # B, M, 3, H, W
        mask: torch.Tensor,            # B, M  (bool)
        order: Optional[Sequence[int]] = None,
        grl_lambda: float = 1.0,
        need_attn: bool = False,
    ) -> Dict[str, torch.Tensor]:
        B, M = images.shape[0], images.shape[1]
        order = list(range(M)) if order is None else list(order)
        dev = images.device

        h = self.h0.expand(B, -1)
        per_mag: Dict[int, Dict[str, torch.Tensor]] = {}
        route: List[Tuple[int, List[Dict[str, torch.Tensor]]]] = []
        attn_maps: Dict[int, torch.Tensor] = {}

        for m in order:
            mag_id = torch.full((B,), m, dtype=torch.long, device=dev)
            cond_parts = [self.mag_emb(mag_id)]
            if self.state_to_cond is not None:
                cond_parts.append(self.state_to_cond(h))
            cond = torch.cat(cond_parts, dim=-1)

            feat, stats = self.backbone(images[:, m], cond)
            route.append((m, stats))

            pooled = feat.mean(dim=(2, 3))
            z_inv = F.normalize(self.proj_inv(pooled), dim=-1)
            z_spec = F.normalize(self.proj_spec(pooled), dim=-1)

            tokens = self.tok_proj(feat).flatten(2).transpose(1, 2)
            tokens = tokens + self.pos + self.zoom_tok(mag_id).unsqueeze(1)

            h_new, step_logit, utility, attn_w = self.pzr.step(h, tokens)
            keep = mask[:, m].float().unsqueeze(-1)
            h = keep * h_new + (1.0 - keep) * h

            per_mag[m] = {
                "z_inv": z_inv,
                "z_spec": z_spec,
                "step_logit": step_logit,
                "utility": utility * mask[:, m].float(),
            }
            if need_attn:
                attn_maps[m] = attn_w.detach()

        stack = lambda key: torch.stack([per_mag[m][key] for m in range(M)], dim=1)
        z_inv_all, z_spec_all = stack("z_inv"), stack("z_spec")

        out = {
            "logit": self.head(h).squeeze(-1),
            "state": h,
            "z_inv": z_inv_all,
            "z_spec": z_spec_all,
            "step_logits": stack("step_logit"),
            "utility": stack("utility"),
            "mag_logits_inv": self.mag_clf_inv(grad_reverse(z_inv_all, grl_lambda)),
            "mag_logits_spec": self.mag_clf_spec(z_spec_all),
            "route": route,
        }
        if need_attn:
            out["attn"] = attn_maps
        return out


# ---------------------------------------------------------------- losses
def info_nce_patient_safe(za, zb, patient_idx, tau=0.1):
    """Positives on the diagonal; same-patient off-diagonals removed from the
    negative set so the model cannot win by encoding slide/stain identity."""
    za, zb = F.normalize(za.float(), dim=-1), F.normalize(zb.float(), dim=-1)
    sim = za @ zb.t() / tau
    same = patient_idx[:, None] == patient_idx[None, :]
    eye = torch.eye(len(za), dtype=torch.bool, device=za.device)
    sim = sim.masked_fill(same & ~eye, float("-inf"))
    target = torch.arange(len(za), device=za.device)
    return F.cross_entropy(sim, target)


def crossmag_align_loss(z_inv, mask, patient_idx, tau=0.1, min_pairs=4):
    """Every ordered pair of magnifications of the same sample is a positive."""
    M = z_inv.shape[1]
    total, n = z_inv.new_zeros(()), 0
    for a in range(M):
        for b in range(a + 1, M):
            sel = mask[:, a] & mask[:, b]
            if int(sel.sum()) < min_pairs:
                continue
            za, zb, pid = z_inv[sel, a], z_inv[sel, b], patient_idx[sel]
            total = total + 0.5 * (
                info_nce_patient_safe(za, zb, pid, tau) + info_nce_patient_safe(zb, za, pid, tau)
            )
            n += 1
    return total / max(n, 1)


def zoom_ordinal_loss(z_spec, mask, margin=0.05):
    """d(z_40, z_100) < d(z_40, z_200) < d(z_40, z_400) on the scale-specific part."""
    anchor_ok = mask[:, 0]
    d = torch.norm(z_spec[:, 1:] - z_spec[:, :1], dim=-1)          # B, M-1
    valid = mask[:, 1:] & anchor_ok.unsqueeze(1)
    total, n = z_spec.new_zeros(()), 0
    for k in range(d.shape[1] - 1):
        v = valid[:, k] & valid[:, k + 1]
        if int(v.sum()) == 0:
            continue
        total = total + F.relu(d[v, k] - d[v, k + 1] + margin).mean()
        n += 1
    return total / max(n, 1)


def disentangle_loss(z_inv, z_spec, mask):
    """Squared cross-covariance between the invariant and specific subspaces."""
    a, b = z_inv[mask], z_spec[mask]
    if a.shape[0] < 4:
        return z_inv.new_zeros(())
    a = (a - a.mean(0)) / (a.std(0) + 1e-5)
    b = (b - b.mean(0)) / (b.std(0) + 1e-5)
    c = (a.t() @ b) / a.shape[0]
    return c.pow(2).sum() / c.numel()


def mag_adversary_losses(out, mask):
    """z_inv must NOT predict magnification (through GRL); z_spec must."""
    M = out["z_inv"].shape[1]
    tgt = torch.arange(M, device=mask.device).unsqueeze(0).expand(mask.shape[0], -1)
    sel = mask.reshape(-1)
    li = F.cross_entropy(out["mag_logits_inv"].reshape(-1, M)[sel], tgt.reshape(-1)[sel])
    ls = F.cross_entropy(out["mag_logits_spec"].reshape(-1, M)[sel], tgt.reshape(-1)[sel])
    return li, ls


def supcon_cross_patient(h, labels, patient_idx, tau=0.1):
    """Positives are same-class, DIFFERENT-patient. Same-patient pairs are
    excluded entirely: too easy as positives, misleading as negatives."""
    z = F.normalize(h.float(), dim=-1)
    sim = z @ z.t() / tau
    n = len(z)
    eye = torch.eye(n, dtype=torch.bool, device=z.device)
    same_pat = patient_idx[:, None] == patient_idx[None, :]
    valid = ~eye & ~same_pat
    pos = valid & (labels[:, None] == labels[None, :])

    logits = sim.masked_fill(~valid, torch.finfo(sim.dtype).min / 4)
    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    cnt = pos.sum(1)
    keep = cnt > 0
    if int(keep.sum()) == 0:
        return h.new_zeros(())
    loss = -(log_prob * pos).sum(1)[keep] / cnt[keep]
    return loss.mean()


def routing_losses(route, n_experts, device=None):
    """
    L_bal   : Switch-style, minimised at uniform marginal expert usage.
    L_spec  : -I(E;M), so usage is informative about magnification.
    L_zloss : router logit magnitude, for numerical stability.
    """
    per_mag = defaultdict(list)
    bal, zl, nb = None, None, 0
    for m, blocks in route:
        for s in blocks:
            term = n_experts * (s["frac"] * s["gates"]).sum()
            bal = term if bal is None else bal + term
            zl = s["zloss"] if zl is None else zl + s["zloss"]
            per_mag[m].append(s["gates"])
            nb += 1
    if nb == 0:  # MoE disabled (ablation)
        zero = torch.zeros((), device=device)
        return zero, zero, zero

    q_m = torch.stack([torch.stack(v).mean(0) for v in per_mag.values()])
    q_m = q_m / q_m.sum(dim=-1, keepdim=True).clamp(min=1e-8)
    q = q_m.mean(0)
    q = q / q.sum().clamp(min=1e-8)
    ent = lambda p: -(p * (p + 1e-9).log()).sum(-1)
    mi = ent(q) - ent(q_m).mean()
    return bal / nb, -mi, zl / nb


def expert_usage_table(route, n_experts, n_mags=N_MAGS):
    """n_mags x n_experts gate-mass table, for the interpretability heatmap."""
    tab = torch.zeros(n_mags, n_experts)
    cnt = torch.zeros(n_mags)
    for m, blocks in route:
        for s in blocks:
            tab[m] += s["gates"].detach().float().cpu()
            cnt[m] += 1
    return tab / cnt.clamp(min=1).unsqueeze(1)


# ---------------------------------------------------------------- objective
def compute_losses(out, batch, cfg: ZoomMoEConfig, supervised=True, unsupervised=True):
    mask = batch["mask"]
    y = batch["label"]
    pid = batch["patient_idx"]
    parts: Dict[str, torch.Tensor] = {}

    if supervised:
        parts["cls"] = F.binary_cross_entropy_with_logits(out["logit"], y)
        valid = mask.reshape(-1)
        deep_tgt = y.unsqueeze(1).expand_as(out["step_logits"]).reshape(-1)[valid]
        parts["deep"] = cfg.w_deep * F.binary_cross_entropy_with_logits(
            out["step_logits"].reshape(-1)[valid], deep_tgt
        )
        parts["supcon"] = cfg.w_supcon * supcon_cross_patient(out["state"], y, pid, cfg.tau)

    if unsupervised:
        parts["align"] = cfg.w_align * crossmag_align_loss(out["z_inv"], mask, pid, cfg.tau)
        parts["ordinal"] = cfg.w_ordinal * zoom_ordinal_loss(out["z_spec"], mask, cfg.ordinal_margin)
        parts["disent"] = cfg.w_disent * disentangle_loss(out["z_inv"], out["z_spec"], mask)
        li, ls = mag_adversary_losses(out, mask)
        parts["mag_adv"] = cfg.w_advmag * li
        parts["mag_spec"] = cfg.w_advmag * ls

    bal, spec, zl = routing_losses(out["route"], cfg.n_experts, out["logit"].device)
    parts["bal"] = cfg.w_bal * bal
    parts["moe_spec"] = cfg.w_spec * spec
    parts["zloss"] = cfg.w_zloss * zl

    total = sum(parts.values())
    return total, {k: float(v.detach()) for k, v in parts.items()}


# ---------------------------------------------------------------- inference helper
@torch.no_grad()
def collect(model, loader, device, order=None, max_batches=None):
    """Run the model over a loader and return everything notebook 03 needs."""
    model.eval()
    acc = defaultdict(list)
    usage, n_batch = None, 0
    for i, batch in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        images = batch["images"].to(device)
        mask = batch["mask"].to(device)
        out = model(images, mask, order=order)
        acc["prob"].append(torch.sigmoid(out["logit"]).cpu().numpy())
        acc["step_prob"].append(torch.sigmoid(out["step_logits"]).cpu().numpy())
        acc["utility"].append(out["utility"].cpu().numpy())
        acc["state"].append(out["state"].cpu().numpy())
        acc["z_inv"].append(out["z_inv"].cpu().numpy())
        acc["z_spec"].append(out["z_spec"].cpu().numpy())
        acc["label"].append(batch["label"].numpy())
        acc["patient_idx"].append(batch["patient_idx"].numpy())
        acc["mask"].append(batch["mask"].numpy())
        tab = expert_usage_table(out["route"], model.cfg.n_experts)
        usage = tab if usage is None else usage + tab
        n_batch += 1
    result = {k: np.concatenate(v) for k, v in acc.items()}
    result["expert_usage"] = (usage / max(n_batch, 1)).numpy()
    return result

Writing zoommoe_model.py


In [3]:
import importlib
import zoommoe_model as zm

importlib.reload(zm)
print("model module loaded")

model module loaded


## 2. Smoke test on random tensors

Ten seconds, no dataset, no pretrained download. Run this first — it catches
every shape bug before you spend GPU hours.

In [4]:
def smoke_test():
    cfg = zm.ZoomMoEConfig(img_size=64, n_experts=4, top_k=2, pretrained=False, state_dim=64,
                           tok_dim=64, proj_dim=32, n_queries=4)
    model = zm.ZoomMoE(cfg)
    B, M = 6, 4
    images = torch.randn(B, M, 3, 64, 64)
    mask = torch.ones(B, M, dtype=torch.bool)
    mask[0, 3] = False          # missing magnification
    mask[1, 1] = False
    batch = {
        "images": images,
        "mask": mask,
        "label": torch.randint(0, 2, (B,)).float(),
        "patient_idx": torch.tensor([0, 0, 1, 1, 2, 3]),
    }

    out = model(images, mask)
    for k in ("logit", "state", "z_inv", "z_spec", "step_logits", "utility"):
        print(f"  {k:12s} {tuple(out[k].shape)}")

    total, parts = zm.compute_losses(out, batch, cfg)
    print("  loss terms:", {k: round(v, 4) for k, v in parts.items()})
    total.backward()

    grads = [p.grad is not None for p in model.parameters() if p.requires_grad]
    print(f"  parameters receiving gradient: {sum(grads)}/{len(grads)}")
    assert torch.isfinite(total), "loss is not finite"

    # permuted order must run and must produce a different state
    out_p = model(images, mask, order=[3, 2, 1, 0])
    delta = (out["state"] - out_p["state"]).abs().mean().item()
    print(f"  mean |state - state_permuted| = {delta:.5f}  (should be > 0)")

    usage = zm.expert_usage_table(out["route"], cfg.n_experts)
    print(f"  expert usage table {tuple(usage.shape)}, row sums {usage.sum(1).tolist()}")
    print("  params:", sum(p.numel() for p in model.parameters()))
    return model


_ = smoke_test()
del _
print("\nsmoke test passed")

  logit        (6,)
  state        (6, 64)
  z_inv        (6, 4, 32)
  z_spec       (6, 4, 32)
  step_logits  (6, 4)
  utility      (6, 4)
  loss terms: {'cls': 0.7137, 'deep': 0.2084, 'supcon': 0.7348, 'align': 1.5209, 'ordinal': 0.0348, 'disent': 0.0242, 'mag_adv': 0.2771, 'mag_spec': 0.2806, 'bal': 0.0209, 'moe_spec': -0.0003, 'zloss': 0.0025}
  parameters receiving gradient: 237/237
  mean |state - state_permuted| = 0.03082  (should be > 0)
  expert usage table (4, 4), row sums [1.0, 0.9999999403953552, 1.0, 1.0]
  params: 36366674

smoke test passed


## 3. Load folds from notebook 01

In [5]:
samples = pd.read_pickle("/kaggle/input/datasets/nabinkoirala452/articafts/artifacts/samples_pool.pkl")   # pooled pairing = free augmentation
samples_ref = pd.read_pickle("/kaggle/input/datasets/nabinkoirala452/articafts/artifacts/samples_all.pkl")
folds = list(np.load("/kaggle/input/datasets/nabinkoirala452/articafts/artifacts/folds_all.npy", allow_pickle=True))

# folds were built on samples_all; the two tables must be row-aligned
assert (samples.sample_id.values == samples_ref.sample_id.values).all(), \
    "sample tables are not aligned - rebuild both in notebook 01"
bc.audit_folds(samples, folds)

QUICK = False         # subsample patients for a fast pipeline check
N_FOLDS = 1 if QUICK else 5

if QUICK:
    keep = samples.patient_id.isin(sorted(samples.patient_id.unique())[:20])
    print(f"QUICK mode: {int(keep.sum())} of {len(samples)} samples, 20 patients")

fold 0: n={'train': 1638, 'val': 308, 'test': 461}  patients={'train': 56, 'val': 10, 'test': 16}  malignant frac={'train': 0.715, 'val': 0.529, 'test': 0.753}
fold 1: n={'train': 1707, 'val': 309, 'test': 391}  patients={'train': 56, 'val': 10, 'test': 16}  malignant frac={'train': 0.733, 'val': 0.443, 'test': 0.749}
fold 2: n={'train': 1622, 'val': 283, 'test': 502}  patients={'train': 56, 'val': 10, 'test': 16}  malignant frac={'train': 0.709, 'val': 0.47, 'test': 0.793}
fold 3: n={'train': 1652, 'val': 233, 'test': 522}  patients={'train': 55, 'val': 10, 'test': 17}  malignant frac={'train': 0.712, 'val': 0.545, 'test': 0.722}
fold 4: n={'train': 1608, 'val': 268, 'test': 531}  patients={'train': 55, 'val': 10, 'test': 17}  malignant frac={'train': 0.774, 'val': 0.634, 'test': 0.501}


## 4. Staged training

| stage | trains | losses | why |
|---|---|---|---|
| 0 | backbone + MoE + projection heads | align, ordinal, disentangle, routing | experts differentiate before any class signal can bias them; uses all images, no labels |
| 1 | PZR + heads, early backbone frozen | cls, deep, supcon (+ small unsupervised) | fit the classifier on stable features |
| 2 | everything, low LR | all | joint refinement, early-exit calibration |

In [6]:
from contextlib import nullcontext

from torch.utils.data import DataLoader


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def autocast_ctx(enabled):
    if not enabled:
        return nullcontext()
    try:
        return torch.amp.autocast("cuda")
    except (AttributeError, TypeError):
        return torch.cuda.amp.autocast()


def make_loader(idx, train, batch_size, scale_dropout=0.0, img_size=224, workers=2):
    ds = bc.make_image_dataset(
        samples.iloc[idx], train=train, img_size=img_size,
        scale_dropout=scale_dropout, seed=SEED,
    )
    return DataLoader(
        ds, batch_size=batch_size, shuffle=train, num_workers=workers,
        pin_memory=True, drop_last=train,
    )


def run_stage(model, loader, cfg, epochs, lr, supervised, unsupervised,
              freeze=None, grl_ramp=True, amp=True, log_every=20, tag=""):
    model.backbone.freeze_until(freeze)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=lr, total_steps=max(epochs * len(loader), 1), pct_start=0.3
    )
    use_amp = amp and DEVICE.type == "cuda"
    scaler = make_scaler(use_amp)

    for ep in range(epochs):
        model.train()
        running = []
        for it, batch in enumerate(loader):
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            frac = (ep * len(loader) + it) / max(epochs * len(loader), 1)
            lam = float(2.0 / (1.0 + np.exp(-10 * frac)) - 1.0) if grl_ramp else 1.0

            with autocast_ctx(use_amp):
                out = model(batch["images"], batch["mask"], grl_lambda=lam)
                total, parts = zm.compute_losses(
                    out, batch, cfg, supervised=supervised, unsupervised=unsupervised
                )

            opt.zero_grad(set_to_none=True)
            scaler.scale(total).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 5.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            running.append(parts)

            if it % log_every == 0:
                msg = "  ".join(f"{k}={v:.3f}" for k, v in parts.items())
                print(f"  [{tag}] ep{ep} it{it}: total={float(total):.3f}  {msg}", flush=True)

        if not running:
            print(f"  [{tag}] epoch {ep}: empty loader, nothing to train on")
            continue
        mean = {k: float(np.mean([r[k] for r in running])) for k in running[0]}
        print(f"  [{tag}] epoch {ep} mean: " + "  ".join(f"{k}={v:.3f}" for k, v in mean.items()))
    return model


@torch.no_grad()
def evaluate_loader(model, loader):
    res = zm.collect(model, loader, DEVICE)
    m = bc.evaluate(res["label"], res["prob"], res["patient_idx"])
    return m, res

In [7]:
import sys

# 1. The read-only path we want Python to STOP looking at
bad_path = "/kaggle/input/datasets/nabinkoirala452/code-files"

# 2. Remove it from sys.path so Python can't find the old file
while bad_path in sys.path:
    sys.path.remove(bad_path)

# 3. Add the working directory (where our fixed file is) to the front
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

# 4. Brutally delete the cached module from Python's memory
if 'zoommoe_model' in sys.modules:
    del sys.modules['zoommoe_model']

# 5. Re-import it. It has no choice but to grab the new one now!
import zoommoe_model as zm

print("Now using the model file from:")
print(zm.__file__)

Now using the model file from:
/kaggle/working/zoommoe_model.py


In [8]:
BATCH = 8               # 4 backbone passes per sample; raise if memory allows
EPOCHS_S0, EPOCHS_S1, EPOCHS_S2 = (1, 2, 1) if QUICK else (8, 15, 8)
IMG = 224

cfg = zm.ZoomMoEConfig(img_size=IMG, n_experts=8, top_k=2, pretrained=True)
fold_results, histories = [], {}

for fi in range(N_FOLDS):
    fold = folds[fi]
    if QUICK:
        fold = {k: np.array([i for i in v if keep.iloc[i]]) for k, v in fold.items()}
    print(f"\n================ fold {fi} ================")
    print({k: len(v) for k, v in fold.items()})
    bc.set_seed(SEED + fi)

    model = zm.ZoomMoE(cfg).to(DEVICE)
    print("trainable params:", bc.count_params(model))

    ssl_loader = make_loader(fold["train"], True, BATCH, scale_dropout=0.15, img_size=IMG)
    sup_loader = make_loader(fold["train"], True, BATCH, scale_dropout=0.15, img_size=IMG)
    val_loader = make_loader(fold["val"], False, BATCH, img_size=IMG)
    test_loader = make_loader(fold["test"], False, BATCH, img_size=IMG)

    # stage 0 — unsupervised warm-up
    run_stage(model, ssl_loader, cfg, EPOCHS_S0, 3e-4,
              supervised=False, unsupervised=True, freeze="layer2", tag=f"f{fi}-s0")

    # stage 1 — supervised fusion
    run_stage(model, sup_loader, cfg, EPOCHS_S1, 3e-4,
              supervised=True, unsupervised=True, freeze="layer2", tag=f"f{fi}-s1")
    vm, _ = evaluate_loader(model, val_loader)
    print(f"  [f{fi}] after stage 1, val:", {k: round(v, 4) for k, v in vm.items()})

    # stage 2 — end-to-end fine-tune
    run_stage(model, sup_loader, cfg, EPOCHS_S2, 3e-5,
              supervised=True, unsupervised=True, freeze=None, tag=f"f{fi}-s2")

    vm, _ = evaluate_loader(model, val_loader)
    tm, test_res = evaluate_loader(model, test_loader)
    print(f"  [f{fi}] val :", {k: round(v, 4) for k, v in vm.items()})
    print(f"  [f{fi}] TEST:", {k: round(v, 4) for k, v in tm.items()})

    tm["fold"] = fi
    fold_results.append(tm)
    torch.save({"cfg": cfg, "state_dict": model.state_dict()}, OUT / f"zoommoe_fold{fi}.pt")
    np.savez_compressed(OUT / f"zoommoe_test_fold{fi}.npz", **test_res)
    del model
    torch.cuda.empty_cache()


================ fold 0 ================
{'train': 1638, 'val': 308, 'test': 461}
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 161MB/s]


trainable params: 48538906
  [f0-s0] ep0 it0: total=2.669  align=2.042  ordinal=0.024  disent=0.020  mag_adv=0.278  mag_spec=0.278  bal=0.023  moe_spec=-0.000  zloss=0.005


/tmp/ipykernel_24/3117464509.py:69: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(f"  [{tag}] ep{ep} it{it}: total={float(total):.3f}  {msg}", flush=True)


  [f0-s0] ep0 it20: total=2.378  align=1.732  ordinal=0.045  disent=0.019  mag_adv=0.279  mag_spec=0.277  bal=0.023  moe_spec=-0.000  zloss=0.005
  [f0-s0] ep0 it40: total=2.067  align=1.422  ordinal=0.041  disent=0.021  mag_adv=0.277  mag_spec=0.278  bal=0.024  moe_spec=-0.001  zloss=0.005
  [f0-s0] ep0 it60: total=2.215  align=1.572  ordinal=0.041  disent=0.020  mag_adv=0.278  mag_spec=0.277  bal=0.023  moe_spec=-0.001  zloss=0.005
  [f0-s0] ep0 it80: total=2.265  align=1.630  ordinal=0.035  disent=0.019  mag_adv=0.278  mag_spec=0.277  bal=0.023  moe_spec=-0.001  zloss=0.004
  [f0-s0] ep0 it100: total=1.967  align=1.355  ordinal=0.009  disent=0.021  mag_adv=0.278  mag_spec=0.278  bal=0.023  moe_spec=-0.001  zloss=0.004
  [f0-s0] ep0 it120: total=1.555  align=0.902  ordinal=0.050  disent=0.023  mag_adv=0.277  mag_spec=0.276  bal=0.024  moe_spec=-0.001  zloss=0.004
  [f0-s0] ep0 it140: total=1.423  align=0.792  ordinal=0.023  disent=0.024  mag_adv=0.278  mag_spec=0.278  bal=0.024  moe_

...

  [f1-s1] ep0 it20: total=2.442  cls=0.530  deep=0.343  supcon=1.225  align=0.009  ordinal=0.000  disent=0.002  mag_adv=0.278  mag_spec=0.018  bal=0.052  moe_spec=-0.018  zloss=0.003
  [f1-s1] ep0 it40: total=2.945  cls=0.793  deep=0.216  supcon=1.556  align=0.040  ordinal=0.000  disent=0.006  mag_adv=0.278  mag_spec=0.019  bal=0.051  moe_spec=-0.016  zloss=0.003
  [f1-s1] ep0 it60: total=2.477  cls=0.616  deep=0.211  supcon=1.304  align=0.009  ordinal=0.000  disent=0.004  mag_adv=0.278  mag_spec=0.019  bal=0.051  moe_spec=-0.018  zloss=0.003
  [f1-s1] ep0 it80: total=3.070  cls=0.633  deep=0.168  supcon=1.881  align=0.054  ordinal=0.000  disent=0.003  mag_adv=0.277  mag_spec=0.019  bal=0.050  moe_spec=-0.017  zloss=0.003
  [f1-s1] ep0 it100: total=2.523  cls=0.524  deep=0.154  supcon=1.494  align=0.014  ordinal=0.000  disent=0.004  mag_adv=0.278  mag_spec=0.018  bal=0.053  moe_spec=-0.018  zloss=0.003
  [f1-s1] ep0 it120: total=2.347  cls=0.500  deep=0.103  supcon=1.371  align=0.032  

/tmp/ipykernel_24/3117464509.py:64: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


  [f2-s1] ep0 it20: total=3.483  cls=0.755  deep=0.441  supcon=1.959  align=0.006  ordinal=0.000  disent=0.002  mag_adv=0.277  mag_spec=0.018  bal=0.044  moe_spec=-0.023  zloss=0.003
  [f2-s1] ep0 it40: total=3.753  cls=0.762  deep=0.296  supcon=2.342  align=0.028  ordinal=0.000  disent=0.004  mag_adv=0.278  mag_spec=0.017  bal=0.046  moe_spec=-0.023  zloss=0.003
  [f2-s1] ep0 it60: total=2.612  cls=0.552  deep=0.139  supcon=1.595  align=0.003  ordinal=0.000  disent=0.003  mag_adv=0.278  mag_spec=0.018  bal=0.045  moe_spec=-0.023  zloss=0.003
  [f2-s1] ep0 it80: total=3.223  cls=0.663  deep=0.128  supcon=1.876  align=0.232  ordinal=0.000  disent=0.002  mag_adv=0.277  mag_spec=0.018  bal=0.047  moe_spec=-0.024  zloss=0.003
  [f2-s1] ep0 it100: total=2.313  cls=0.678  deep=0.106  supcon=1.188  align=0.015  ordinal=0.000  disent=0.004  mag_adv=0.278  mag_spec=0.018  bal=0.047  moe_spec=-0.023  zloss=0.003
  [f2-s1] ep0 it120: total=2.743  cls=0.454  deep=0.133  supcon=1.687  align=0.133  

/tmp/ipykernel_24/3117464509.py:64: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


  [f4-s1] ep0 it20: total=2.871  cls=0.630  deep=0.212  supcon=1.665  align=0.036  ordinal=0.000  disent=0.005  mag_adv=0.277  mag_spec=0.018  bal=0.040  moe_spec=-0.014  zloss=0.003
  [f4-s1] ep0 it40: total=3.380  cls=0.764  deep=0.226  supcon=2.028  align=0.034  ordinal=0.000  disent=0.006  mag_adv=0.276  mag_spec=0.017  bal=0.040  moe_spec=-0.014  zloss=0.003
  [f4-s1] ep0 it60: total=2.879  cls=0.702  deep=0.268  supcon=1.578  align=0.003  ordinal=0.000  disent=0.004  mag_adv=0.278  mag_spec=0.018  bal=0.040  moe_spec=-0.014  zloss=0.003
  [f4-s1] ep0 it80: total=3.028  cls=0.611  deep=0.177  supcon=1.808  align=0.104  ordinal=0.000  disent=0.002  mag_adv=0.277  mag_spec=0.018  bal=0.041  moe_spec=-0.014  zloss=0.003
  [f4-s1] ep0 it100: total=3.180  cls=0.553  deep=0.140  supcon=2.059  align=0.100  ordinal=0.000  disent=0.004  mag_adv=0.276  mag_spec=0.017  bal=0.040  moe_spec=-0.013  zloss=0.003
  [f4-s1] ep0 it120: total=3.185  cls=0.686  deep=0.176  supcon=1.987  align=0.002  

In [9]:
summary = bc.summarise_folds(fold_results)
display(summary.round(4))
pd.DataFrame(fold_results).to_csv(OUT / "zoommoe_results.csv", index=False)

try:
    base = pd.read_csv(OUT / "baseline_results_honest.csv", index_col=0)
    print("\nbaselines from notebook 01:")
    display(base)
except FileNotFoundError:
    print("run notebook 01 first for the baseline comparison")

,mean,std
acc,0.8906,0.1404
f1_macro,0.8657,0.1574
prr,0.9022,0.0855
auc,0.9731,0.0271
fold,2.0000,1.5811


run notebook 01 first for the baseline comparison


## 5. Ablation driver

One switch per claim. Each row removes exactly one thing, so a drop is
attributable. Run with `QUICK = False` and a single fold first; the full grid
across five folds is expensive.

In [10]:
ABLATIONS = {
    "full": {},
    "no_moe": dict(moe_stages=()),
    "moe_mag_only": dict(cond_state_dim=0),        # router loses h_{k-1}
    "no_contrastive": dict(w_align=0.0, w_ordinal=0.0, w_disent=0.0, w_advmag=0.0, w_supcon=0.0),
    "no_ordinal": dict(w_ordinal=0.0),
    "no_supcon": dict(w_supcon=0.0),
    "no_moe_spec_loss": dict(w_spec=0.0),
    "experts_4": dict(n_experts=4),
    "experts_16": dict(n_experts=16),
    "topk_1": dict(top_k=1),
}


def run_ablation(name, overrides, fold, epochs=(2, 4, 2), batch=BATCH):
    from dataclasses import replace

    acfg = replace(cfg, **overrides)
    bc.set_seed(SEED)
    model = zm.ZoomMoE(acfg).to(DEVICE)
    tr = make_loader(fold["train"], True, batch, scale_dropout=0.15, img_size=acfg.img_size)
    te = make_loader(fold["test"], False, batch, img_size=acfg.img_size)
    run_stage(model, tr, acfg, epochs[0], 3e-4, False, True, freeze="layer2", tag=f"{name}-s0")
    run_stage(model, tr, acfg, epochs[1], 3e-4, True, True, freeze="layer2", tag=f"{name}-s1")
    run_stage(model, tr, acfg, epochs[2], 3e-5, True, True, freeze=None, tag=f"{name}-s2")
    m, _ = evaluate_loader(model, te)
    m["ablation"] = name
    m["params"] = bc.count_params(model)
    del model
    torch.cuda.empty_cache()
    return m


RUN_ABLATIONS = False   # flip on once the full model trains cleanly

if RUN_ABLATIONS:
    fold = folds[0]
    if QUICK:
        fold = {k: np.array([i for i in v if keep.iloc[i]]) for k, v in fold.items()}
    rows = [run_ablation(n, o, fold) for n, o in ABLATIONS.items()]
    ab = pd.DataFrame(rows).set_index("ablation")
    display(ab.round(4))
    ab.to_csv(OUT / "ablations.csv")

### Note on `moe_mag_only`

`cond_state_dim=0` makes `state_to_cond` emit a zero-width vector, so the
router sees only the local feature and the magnification embedding. That is
the standard conditional-MoE setup, and the gap between it and `full` isolates
the contribution of the zoom-state feedback edge — the single most important
number in the ablation table.

## 6. Handoff

Written to `artifacts/`: `zoommoe_fold*.pt`, `zoommoe_test_fold*.npz`,
`zoommoe_results.csv`, and `zoommoe_model.py`.

Next: `03_analysis_and_falsification.ipynb`, which runs the tests that could
show this whole direction does nothing.